In [265]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import numpy as np

In [266]:
# SETUP 

TRAINING_DATA = "training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id', 
    'best_tasks_free', 
    'acad_tasks_rating', 
    'best_tasks_select', 
    'subopt_freq_rating',  
    'subopt_tasks_select', 
    'subopt_tasks_free', 
    'evidence_freq_rating', 
    'verify_freq_rating', 
    'verify_method_free', 
    'target'
    ]

text_cols = [
    'best_tasks_free', 
    'subopt_tasks_free', 
    'verify_method_free',
    ] 


ord_cols = [
    'acad_tasks_rating', 
    'subopt_freq_rating',  
    'evidence_freq_rating', 
    'verify_freq_rating', 
    ]

acad_categories = [
    '1 — Not at all likely',
    '2 — Unlikely',
    '3 — Neutral / Unsure',
    '4 — Likely',
    '5 — Very likely'
]

freq_categories = [
    '1 — Never',
    '2 — Rarely',
    '3 — Sometimes',
    '4 — Often',
    '5 — Very often'
]

ord_categories_per_col = {
    'acad_tasks_rating': acad_categories,
    'subopt_freq_rating': freq_categories,
    'evidence_freq_rating': freq_categories,
    'verify_freq_rating': freq_categories,
}

cat_cols = [
    'best_tasks_select', 
    'subopt_tasks_select',
    ]

cat_multi_select_choices = [
    "Explaining complex concepts simply",
    "Drafting professional text (e.g., emails, résumés)",
    "Brainstorming or generating creative ideas",
    "Writing or editing essays/reports",
    "Converting content between formats (e.g., LaTeX)",
    "Writing or debugging code",
    "Data processing or analysis",
    "Math computations",
]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [267]:
# Implementation of TFIDF

import re
from collections import Counter

STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "at",
    "with", "is", "it", "this", "that", "as", "by", "be", "are", "was",
    "were", "from", "but", "if", "so", "not", "isn", "t"
}

TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)

def tokenize(text, lower=True, remove_stopwords=True, stopwords=STOPWORDS):
    if not isinstance(text, str):
        return []
    if lower:
        text = text.lower()
    tokens = TOKEN_PATTERN.findall(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

def generate_ngrams(tokens, ngram_range=(1, 1)):
    min_n, max_n = ngram_range
    L = len(tokens)
    all_ngrams = []
    for n in range(min_n, max_n + 1):
        if L < n:
            continue
        for i in range(L - n + 1):
            all_ngrams.append(" ".join(tokens[i:i+n]))
    return all_ngrams

def fit_tfidf(
    corpus,
    ngram_range=(1, 3),
    min_df=1,
    max_df=1.0,
    max_features=None,
    remove_stopwords=True,
):
    """
    TF-IDF formula:
      - tokenization + n-grams
      - df filtering with min_df / max_df
      - idf(t) = log((1 + N) / (1 + df)) + 1
      - L2-normalization per document
    """
    N = len(corpus)
    doc_terms = []
    df_counter = Counter()

    for doc in corpus:
        tokens = tokenize(doc, remove_stopwords=remove_stopwords)
        terms = generate_ngrams(tokens, ngram_range=ngram_range)
        doc_terms.append(terms)

        unique_terms = set(terms)
        for t in unique_terms:
            df_counter[t] += 1

    vocab_terms = []
    for term, df in df_counter.items():
        if df < min_df:
            continue
        if isinstance(max_df, float) and max_df <= 1.0:
            if df > max_df * N:
                continue
        else:
            if df > max_df:
                continue
        vocab_terms.append(term)

    if max_features is not None:
        vocab_terms = sorted(
            vocab_terms,
            key=lambda t: df_counter[t],
            reverse=True
        )[:max_features]
    else:
        vocab_terms = sorted(vocab_terms)

    vocab = {term: j for j, term in enumerate(vocab_terms)}
    V = len(vocab)

    idf = np.zeros(V, dtype=np.float64)
    for term, j in vocab.items():
        df = df_counter[term]
        idf[j] = np.log((1.0 + N) / (1.0 + df)) + 1.0

    X_tfidf = np.zeros((N, V), dtype=np.float64)
    for i, terms in enumerate(doc_terms):
        if not terms:
            continue
        tf_counts = Counter(t for t in terms if t in vocab)
        if not tf_counts:
            continue
        doc_len = sum(tf_counts.values())
        for term, cnt in tf_counts.items():
            j = vocab[term]
            tf = cnt / doc_len
            X_tfidf[i, j] = tf * idf[j]

    norms = np.linalg.norm(X_tfidf, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    X_tfidf = X_tfidf / norms

    return X_tfidf, vocab, idf


In [268]:
TEXT_TFIDF_CONFIG = {
    "best_tasks_free": {
        "ngram_range": (1, 2),
        "max_features": 1200,
        "min_df": 2,
        "max_df": 0.75,
    },
    "subopt_tasks_free": {
        "ngram_range": (1, 2),
        "max_features": 800,
        "min_df": 2,
        "max_df": 0.75,
    },
    "verify_method_free": {
        "ngram_range": (1, 1),
        "max_features": 500,
        "min_df": 2,
        "max_df": 0.75,
    },
}

def make_text_corpus(df, col):
    """
    Return a list of strings for a given text column,
    with NaNs replaced by empty strings.
    """
    return ["" if pd.isna(x) else str(x) for x in df[col]]

def fit_tfidf_separate_columns(df, text_config):
    """
    Fit one manual TF-IDF per text column and horizontally stack the results.
    
    Returns:
      X_text : (N, sum_of_column_feature_dims) numpy array
      tfidf_models : dict mapping col_name -> dict with vocab, idf, and params
    """
    N = len(df)
    X_blocks = []
    tfidf_models = {}

    for col, cfg in text_config.items():
        corpus = make_text_corpus(df, col)

        X_col, vocab, idf = fit_tfidf(
            corpus,
            ngram_range=cfg.get("ngram_range", (1, 1)),
            min_df=cfg.get("min_df", 1),
            max_df=cfg.get("max_df", 1.0),
            max_features=cfg.get("max_features", None),
            remove_stopwords=cfg.get("remove_stopwords", True),
        )

        print(f"[{col}] TF-IDF shape: {X_col.shape}, vocab size: {len(vocab)}")

        X_blocks.append(X_col)

        tfidf_models[col] = {
            "vocab": vocab,
            "idf": idf,
            "ngram_range": cfg.get("ngram_range", (1, 1)),
            "min_df": cfg.get("min_df", 1),
            "max_df": cfg.get("max_df", 1.0),
            "max_features": cfg.get("max_features", None),
            "remove_stopwords": cfg.get("remove_stopwords", True),
        }

    # Horizontally stack: [best_tasks | subopt_tasks | verify_method]
    X_text = np.hstack(X_blocks)
    print("Combined text TF-IDF shape:", X_text.shape)

    return X_text, tfidf_models


In [269]:
# Ordinal data preprocessing

from collections import Counter
import numpy as np
import pandas as pd

# Create mappings and fill values(frequency based)
def fit_ordinal_encoders(df, ord_cols, ord_categories_per_col):
    """
    Build per-column mappings and per-column *median*-based fill values.

    ord_categories_per_col: dict {col_name: [ordered categories for that column]}
    """
    mappings = {}
    fill_values = {}

    for col in ord_cols:
        # 1) Ordered categories for this column only
        cats = ord_categories_per_col[col]
        mapping = {cat: i for i, cat in enumerate(cats)}
        mappings[col] = mapping

        # 2) Get valid (non-NaN, known) values in this column
        col_vals = df[col].dropna()
        encoded_vals = [mapping[v] for v in col_vals if v in mapping]

        if encoded_vals:
            # 3) Compute the *column-specific* median index
            median_idx = int(np.median(encoded_vals))
            fill_values[col] = median_idx
        else:
            # 4) Fallback: middle category if column is empty/weird
            fill_values[col] = len(cats) // 2

    return mappings, fill_values

# Trnasform data from dataset using mappings
def transform_ordinal(df, ord_cols, mappings, fill_values):
    N = len(df)
    K = len(ord_cols)
    X_ord = np.zeros((N, K), dtype=np.float64)

    for j, col in enumerate(ord_cols):
        mapping = mappings[col]
        fill_val = fill_values[col]

        encoded_col = []
        for v in df[col].tolist():
            if pd.isna(v) or v not in mapping:
                encoded_col.append(fill_val)
            else:
                encoded_col.append(mapping[v])

        X_ord[:, j] = np.array(encoded_col, dtype=np.float64)

    return X_ord

In [270]:
# Categorical preprocessing(multi select)

# parse text from multiselect questions to get categories selected
def parse_multiselect(raw_value, choice_list):
    if pd.isna(raw_value):
        return []  # skipped → no selections

    txt = str(raw_value).strip()
    if txt == "":
        return []

    selections = []
    for choice in choice_list:
        if choice in txt:
            selections.append(choice)

    return selections

# Transform categorical data to one-hot encoding for mult-select categorical data
def transform_multiselect_categorical(df, cat_cols, cat_multi_select_choices):
    N = len(df)
    M = len(cat_multi_select_choices)

    # total features = (#cat_cols) * (#choices)
    X = np.zeros((N, len(cat_cols) * M), dtype=np.float32)
    feature_names = []

    # build feature name list in the same order we fill X
    for col in cat_cols:
        for choice in cat_multi_select_choices:
            feature_names.append(f"{col}__{choice}")

    for i in range(N):
        offset = 0
        for col in cat_cols:
            selections = parse_multiselect(df.loc[i, col], cat_multi_select_choices)
            for j, choice in enumerate(cat_multi_select_choices):
                X[i, offset + j] = 1.0 if choice in selections else 0.0
            offset += M

    return X, feature_names


In [271]:
#Combine all features
X_text, tfidf_models = fit_tfidf_separate_columns(df, TEXT_TFIDF_CONFIG)

print("Manual TF-IDF matrix shape:", X_text.shape)
for col, model in tfidf_models.items():
    vocab_size = len(model["vocab"])
    print(f"{col} vocabulary size: {vocab_size}")
total_vocab = sum(len(model["vocab"]) for model in tfidf_models.values())
print("Total combined vocabulary size:", total_vocab)

ord_mappings, ord_fill_values = fit_ordinal_encoders(df, ord_cols, ord_categories_per_col)
X_ord = transform_ordinal(df, ord_cols, ord_mappings, ord_fill_values)

print("Ordinal feature matrix shape:", X_ord.shape)

X_cat, cat_feature_names = transform_multiselect_categorical(
    df,
    cat_cols=cat_cols,
    cat_multi_select_choices=cat_multi_select_choices,
)

print("Categorical (multi-select) feature matrix shape:", X_cat.shape)
print("Number of categorical features:", len(cat_feature_names))

X_all = np.hstack([X_text, X_ord, X_cat])
print("Final combined feature matrix shape:", X_all.shape)


[best_tasks_free] TF-IDF shape: (825, 1200), vocab size: 1200
[subopt_tasks_free] TF-IDF shape: (825, 800), vocab size: 800
[verify_method_free] TF-IDF shape: (825, 500), vocab size: 500
Combined text TF-IDF shape: (825, 2500)
Manual TF-IDF matrix shape: (825, 2500)
best_tasks_free vocabulary size: 1200
subopt_tasks_free vocabulary size: 800
verify_method_free vocabulary size: 500
Total combined vocabulary size: 2500
Ordinal feature matrix shape: (825, 4)
Categorical (multi-select) feature matrix shape: (825, 16)
Number of categorical features: 16
Final combined feature matrix shape: (825, 2520)


In [272]:
# Quick test using all features and sklearn log reg
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# If labels are strings:
labels = df["target"].to_numpy()
unique_classes, y_int = np.unique(labels, return_inverse=True)
print("Class mapping:", dict(enumerate(unique_classes)))
y_all = y_int
num_classes = len(unique_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, labels, test_size=0.2, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train)
test_acc = clf.score(X_test, y_test)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")    

Class mapping: {0: 'ChatGPT', 1: 'Claude', 2: 'Gemini'}
Train accuracy: 0.964
Test accuracy: 0.642


In [273]:
#data split
import numpy as np

def train_val_split(X, y, test_size=0.2, random_state=42, stratify=True):
    """
    Simple train/val split to avoid sklearn.
    X: (N, D)
    y: (N,)
    test_size: fraction for validation split
    stratify: if True, keep class proportions roughly the same
    """
    rng = np.random.default_rng(random_state)
    X = np.asarray(X)
    y = np.asarray(y)
    N = len(y)

    if not stratify:
        idx = rng.permutation(N)
        split = int(N * (1 - test_size))
        train_idx, val_idx = idx[:split], idx[split:]
        return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

    train_idx = []
    val_idx = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        c_idx = rng.permutation(c_idx)
        split = int(len(c_idx) * (1 - test_size))
        train_idx.append(c_idx[:split])
        val_idx.append(c_idx[split:])
    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

In [279]:
#nn implementation
# Adam optimizer
# Cross Entropy Loss
# ReLU activation
# linear output
import numpy as np
from typing import Tuple, Dict

class BatchNorm1d:
    """Batch Normalization layer"""
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable parameters
        self.gamma = np.ones(num_features)
        self.beta = np.zeros(num_features)
        
        # Running statistics for inference
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
        
        # Cache for backward pass
        self.cache = {}
        
    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        if training:
            # Calculate batch statistics
            batch_mean = np.mean(x, axis=0)
            batch_var = np.var(x, axis=0)
            
            # Normalize
            x_normalized = (x - batch_mean) / np.sqrt(batch_var + self.eps)
            
            # Scale and shift
            out = self.gamma * x_normalized + self.beta
            
            # Update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var
            
            # Cache for backward
            self.cache = {
                'x': x,
                'x_normalized': x_normalized,
                'batch_mean': batch_mean,
                'batch_var': batch_var
            }
        else:
            # Use running statistics for inference
            x_normalized = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
            out = self.gamma * x_normalized + self.beta
            
        return out
    
    def backward(self, dout: np.ndarray) -> np.ndarray:
        x = self.cache['x']
        x_normalized = self.cache['x_normalized']
        batch_var = self.cache['batch_var']
        batch_mean = self.cache['batch_mean']
        
        N = x.shape[0]
        
        # Gradients for gamma and beta
        self.dgamma = np.sum(dout * x_normalized, axis=0)
        self.dbeta = np.sum(dout, axis=0)
        
        # Gradient for x
        dx_normalized = dout * self.gamma
        dvar = np.sum(dx_normalized * (x - batch_mean) * -0.5 * (batch_var + self.eps)**(-1.5), axis=0)
        dmean = np.sum(dx_normalized * -1 / np.sqrt(batch_var + self.eps), axis=0) + dvar * np.sum(-2 * (x - batch_mean), axis=0) / N
        dx = dx_normalized / np.sqrt(batch_var + self.eps) + dvar * 2 * (x - batch_mean) / N + dmean / N
        
        return dx


class Dropout:
    """Dropout layer"""
    def __init__(self, p: float = 0.5):
        self.p = p
        self.mask = None
        
    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        if training:
            self.mask = (np.random.rand(*x.shape) > self.p) / (1 - self.p)
            return x * self.mask
        else:
            return x
    
    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * self.mask


class ReLU:
    """ReLU activation function"""
    def __init__(self):
        self.cache = None
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        self.cache = x
        return np.maximum(0, x)
    
    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * (self.cache > 0)


class Linear:
    """Fully connected layer"""
    def __init__(self, in_features: int, out_features: int):
        # He initialization for ReLU
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.b = np.zeros(out_features)
        
        self.cache = None
        self.dW = None
        self.db = None
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        self.cache = x
        return x @ self.W + self.b
    
    def backward(self, dout: np.ndarray) -> np.ndarray:
        x = self.cache
        self.dW = x.T @ dout
        self.db = np.sum(dout, axis=0)
        dx = dout @ self.W.T
        return dx


class AdamOptimizer:
    """Adam optimizer"""
    def __init__(self, lr: float = 0.001, beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = {}  # First moment
        self.v = {}  # Second moment
        
    def step(self, params):
        """params: list of (name, param, grad) tuples"""
        self.t += 1
        
        for name, param, grad in params:
            if name not in self.m:
                self.m[name] = np.zeros_like(param)
                self.v[name] = np.zeros_like(param)
            
            # Update biased first moment estimate
            self.m[name] = self.beta1 * self.m[name] + (1 - self.beta1) * grad
            
            # Update biased second raw moment estimate
            self.v[name] = self.beta2 * self.v[name] + (1 - self.beta2) * (grad ** 2)
            
            # Compute bias-corrected first moment estimate
            m_hat = self.m[name] / (1 - self.beta1 ** self.t)
            
            # Compute bias-corrected second raw moment estimate
            v_hat = self.v[name] / (1 - self.beta2 ** self.t)
            
            # Update parameters
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class NeuralNetwork:
    """One hidden layer neural network with BatchNorm, ReLU, and Dropout"""
    def __init__(self, input_size: int, hidden_units: int, num_classes: int, dropout_rate: float, lr: float):
        self.fc1 = Linear(input_size, hidden_units)
        self.bn1 = BatchNorm1d(hidden_units)
        self.relu = ReLU()
        self.dropout = Dropout(dropout_rate)
        self.fc2 = Linear(hidden_units, num_classes)
        
        self.optimizer = AdamOptimizer(lr=lr)
        
    def forward(self, x: np.ndarray, training: bool = True) -> np.ndarray:
        x = self.fc1.forward(x)
        x = self.bn1.forward(x, training)
        x = self.relu.forward(x)
        x = self.dropout.forward(x, training)
        x = self.fc2.forward(x)
        return x
    
    def backward(self, dout: np.ndarray) -> None:
        dout = self.fc2.backward(dout)
        dout = self.dropout.backward(dout)
        dout = self.relu.backward(dout)
        dout = self.bn1.backward(dout)
        dout = self.fc1.backward(dout)
    
    def update_weights(self):
        params = [
            ('fc1_W', self.fc1.W, self.fc1.dW),
            ('fc1_b', self.fc1.b, self.fc1.db),
            ('bn1_gamma', self.bn1.gamma, self.bn1.dgamma),
            ('bn1_beta', self.bn1.beta, self.bn1.dbeta),
            ('fc2_W', self.fc2.W, self.fc2.dW),
            ('fc2_b', self.fc2.b, self.fc2.db),
        ]
        self.optimizer.step(params)


def softmax(x: np.ndarray) -> np.ndarray:
    """Numerically stable softmax"""
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def cross_entropy_loss(logits: np.ndarray, labels: np.ndarray) -> Tuple[float, np.ndarray]:
    """
    Cross entropy loss with softmax
    labels: integer class labels (not one-hot)
    Returns: (loss, gradient)
    """
    batch_size = logits.shape[0]
    
    # Softmax
    probs = softmax(logits)
    
    # Cross entropy loss
    correct_probs = probs[np.arange(batch_size), labels]
    loss = -np.mean(np.log(correct_probs + 1e-10))
    
    # Gradient
    dlogits = probs.copy()
    dlogits[np.arange(batch_size), labels] -= 1
    dlogits /= batch_size
    
    return loss, dlogits


def create_batches(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool = True):
    """Generate batches from data"""
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    
    if shuffle:
        np.random.shuffle(indices)
    
    for start_idx in range(0, n_samples, batch_size):
        end_idx = min(start_idx + batch_size, n_samples)
        batch_indices = indices[start_idx:end_idx]
        yield X[batch_indices], y[batch_indices]


def train_model(model: NeuralNetwork, X_train: np.ndarray, y_train: np.ndarray, 
                X_val: np.ndarray, y_val: np.ndarray, 
                epochs: int, batch_size: int, patience: int = 3) -> Dict:
    """
    Train the neural network
    
    Args:
        model: Neural network model
        X_train: Training features
        y_train: Training labels (integer encoded)
        X_val: Validation features
        y_val: Validation labels (integer encoded)
        epochs: Number of epochs
        batch_size: Batch size
        patience: Early stopping patience
    
    Returns:
        Dictionary with training history
    """
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        # Training phase
        model_train_loss = 0
        n_batches = 0
        
        for X_batch, y_batch in create_batches(X_train, y_train, batch_size, shuffle=True):
            # Forward pass
            logits = model.forward(X_batch, training=True)
            
            # Compute loss and gradient
            loss, dlogits = cross_entropy_loss(logits, y_batch)
            model_train_loss += loss
            n_batches += 1
            
            # Backward pass
            model.backward(dlogits)
            
            # Update weights
            model.update_weights()
        
        avg_train_loss = model_train_loss / n_batches
        
        # Validation phase
        val_loss, val_acc = evaluate_model(model, X_val, y_val, batch_size)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print("Early stopping triggered")
            break
    
    return history


def evaluate_model(model: NeuralNetwork, X: np.ndarray, y: np.ndarray, batch_size: int) -> Tuple[float, float]:
    """
    Evaluate model on validation/test set
    
    Returns:
        (loss, accuracy)
    """
    total_loss = 0
    correct = 0
    total = 0
    n_batches = 0
    
    for X_batch, y_batch in create_batches(X, y, batch_size, shuffle=False):
        # Forward pass (no training mode)
        logits = model.forward(X_batch, training=False)
        
        # Compute loss
        loss, _ = cross_entropy_loss(logits, y_batch)
        total_loss += loss
        n_batches += 1
        
        # Compute accuracy
        predictions = np.argmax(logits, axis=1)
        correct += np.sum(predictions == y_batch)
        total += len(y_batch)
    
    avg_loss = total_loss / n_batches
    accuracy = correct / total
    
    return avg_loss, accuracy


def grid_search(X_train: np.ndarray, y_train: np.ndarray, 
                X_val: np.ndarray, y_val: np.ndarray,
                input_size: int, num_classes: int,
                param_grid: Dict, epochs: int = 20, patience: int = 3, n_trials: int = 1):
    """
    Perform grid search over hyperparameters with multiple trials per combination
    
    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        input_size: Number of input features
        num_classes: Number of output classes
        param_grid: Dictionary of hyperparameters to search
            Example: {
                'hidden_units': [128, 256, 512],
                'dropout_rate': [0.2, 0.3, 0.4],
                'lr': [0.0001, 0.0003, 0.001],
                'batch_size': [32, 64, 128]
            }
        epochs: Number of epochs per trial
        patience: Early stopping patience
        n_trials: Number of trials to run per combination (for averaging)
    
    Returns:
        List of results sorted by average validation accuracy
    """
    from itertools import product
    
    results = []
    
    # Generate all combinations
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = list(product(*values))
    
    total_runs = len(combinations) * n_trials
    print(f"Testing {len(combinations)} hyperparameter combinations with {n_trials} trial(s) each...")
    print(f"Total training runs: {total_runs}\n")
    
    for i, combo in enumerate(combinations):
        params = dict(zip(keys, combo))
        
        print(f"[{i+1}/{len(combinations)}] Testing: {params}")
        
        trial_accuracies = []
        trial_losses = []
        
        for trial in range(n_trials):
            if n_trials > 1:
                print(f"  Trial {trial+1}/{n_trials}...", end=" ")
            
            # Set seed for reproducibility within trials
            np.random.seed(42 + i * n_trials + trial)
            
            # Create model
            model = NeuralNetwork(
                input_size=input_size,
                hidden_units=params['hidden_units'],
                num_classes=num_classes,
                dropout_rate=params['dropout_rate'],
                lr=params['lr']
            )
            
            # Train model
            history = train_model(
                model=model,
                X_train=X_train,
                y_train=y_train,
                X_val=X_val,
                y_val=y_val,
                epochs=epochs,
                batch_size=params['batch_size'],
                patience=patience
            )
            
            # Get best validation accuracy from this trial
            best_val_acc = max(history['val_acc'])
            best_val_loss = min(history['val_loss'])
            
            trial_accuracies.append(best_val_acc)
            trial_losses.append(best_val_loss)
            
            if n_trials > 1:
                print(f"Acc: {best_val_acc:.4f}")
        
        # Calculate average performance across trials
        avg_val_acc = np.mean(trial_accuracies)
        std_val_acc = np.std(trial_accuracies)
        avg_val_loss = np.mean(trial_losses)
        std_val_loss = np.std(trial_losses)
        
        results.append({
            'params': params,
            'avg_val_acc': avg_val_acc,
            'std_val_acc': std_val_acc,
            'avg_val_loss': avg_val_loss,
            'std_val_loss': std_val_loss,
            'trial_accuracies': trial_accuracies,
            'trial_losses': trial_losses
        })
        
        if n_trials > 1:
            print(f"  Average Val Acc: {avg_val_acc:.4f} ± {std_val_acc:.4f}")
            print(f"  Average Val Loss: {avg_val_loss:.4f} ± {std_val_loss:.4f}\n")
        else:
            print(f"Val Acc: {avg_val_acc:.4f}, Val Loss: {avg_val_loss:.4f}\n")
    
    # Sort by average validation accuracy
    results.sort(key=lambda x: x['avg_val_acc'], reverse=True)
    
    print("\n" + "="*60)
    print("TOP 5 HYPERPARAMETER COMBINATIONS (ranked by average accuracy):")
    print("="*60)
    for i, result in enumerate(results[:5]):
        print(f"\n{i+1}. Params: {result['params']}")
        if n_trials > 1:
            print(f"   Avg Val Acc: {result['avg_val_acc']:.4f} ± {result['std_val_acc']:.4f}")
            print(f"   Avg Val Loss: {result['avg_val_loss']:.4f} ± {result['std_val_loss']:.4f}")
            print(f"   Range: [{min(result['trial_accuracies']):.4f}, {max(result['trial_accuracies']):.4f}]")
        else:
            print(f"   Val Acc: {result['avg_val_acc']:.4f}, Val Loss: {result['avg_val_loss']:.4f}")
    
    return results


def run_multiple_trials(X_train: np.ndarray, y_train: np.ndarray,
                       X_val: np.ndarray, y_val: np.ndarray,
                       input_size: int, num_classes: int,
                       params: Dict, n_trials: int = 10,
                       epochs: int = 30, patience: int = 10):
    """
    Run multiple training trials with the same hyperparameters to assess stability
    
    Args:
        X_train, y_train: Training data
        X_val, y_val: Validation data
        input_size: Number of input features
        num_classes: Number of output classes
        params: Dictionary of hyperparameters (hidden_units, dropout_rate, lr, batch_size)
        n_trials: Number of trials to run
        epochs: Number of epochs per trial
        patience: Early stopping patience
    
    Returns:
        List of results from each trial
    """
    results = []
    
    print(f"Running {n_trials} trials with params: {params}\n")
    
    for trial in range(n_trials):
        print(f"{'='*60}")
        print(f"TRIAL {trial + 1}/{n_trials}")
        print(f"{'='*60}")
        
        np.random.seed(42 + trial)
        
        model = NeuralNetwork(
            input_size=input_size,
            hidden_units=params['hidden_units'],
            num_classes=num_classes,
            dropout_rate=params['dropout_rate'],
            lr=params['lr']
        )
        
        history = train_model(
            model=model,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            epochs=epochs,
            batch_size=params['batch_size'],
            patience=patience
        )
        
        final_val_loss, final_val_acc = evaluate_model(model, X_val, y_val, params['batch_size'])
        results.append({
            'trial': trial + 1,
            'val_acc': final_val_acc,
            'val_loss': final_val_loss,
            'history': history
        })
        
        print(f"Trial {trial + 1} - Val Acc: {final_val_acc:.4f}, Val Loss: {final_val_loss:.4f}\n")
    
    # Calculate statistics
    accuracies = [r['val_acc'] for r in results]
    losses = [r['val_loss'] for r in results]
    
    print(f"{'='*60}")
    print("SUMMARY STATISTICS")
    print(f"{'='*60}")
    print(f"Accuracy:")
    print(f"  Mean: {np.mean(accuracies):.4f}")
    print(f"  Std:  {np.std(accuracies):.4f}")
    print(f"  Min:  {np.min(accuracies):.4f}")
    print(f"  Max:  {np.max(accuracies):.4f}")
    print(f"  Median: {np.median(accuracies):.4f}")
    
    print(f"\nValidation Loss:")
    print(f"  Mean: {np.mean(losses):.4f}")
    print(f"  Std:  {np.std(losses):.4f}")
    
    # Find best trial
    best_trial = max(results, key=lambda x: x['val_acc'])
    print(f"\n{'='*60}")
    print(f"BEST TRIAL: #{best_trial['trial']}")
    print(f"{'='*60}")
    print(f"Accuracy: {best_trial['val_acc']:.4f}")
    print(f"Loss: {best_trial['val_loss']:.4f}")
    
    return results


In [ ]:
#single model test

def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    # If you're using Python's random module anywhere
    import random
    random.seed(seed)

# Before creating model and training
set_seed(42)

X_train, X_val, y_train, y_val = train_val_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=True
)

model = NeuralNetwork(
    input_size=X_train.shape[1],
    hidden_units=512,  # Increased from 256
    num_classes=num_classes,
    dropout_rate=0.1,  # Increased from 0.3
    lr=0.0005  # Increased from 0.0001 (faster learning)
)

print(f"\nTraining with better hyperparameters (no normalization)...")
history = train_model(
    model=model,
    X_train=X_train,  # Original data
    y_train=y_train,
    X_val=X_val,  # Original data
    y_val=y_val,
    epochs=30,  # More epochs
    batch_size=64,  # Smaller batches
    patience=10  # More patience
)

final_val_loss, final_val_acc = evaluate_model(model, X_val, y_val, batch_size=32)
print(f"\nFinal Validation Loss: {final_val_loss:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Improvement over baseline: {final_val_acc - 0.6424:.4f}")


Training with better hyperparameters (no normalization)...
Epoch 1/30, Train Loss: 1.0005, Val Loss: 1.0158, Val Acc: 0.5030
Epoch 2/30, Train Loss: 0.3841, Val Loss: 0.8888, Val Acc: 0.5939
Epoch 3/30, Train Loss: 0.2027, Val Loss: 0.8032, Val Acc: 0.6727
Epoch 4/30, Train Loss: 0.1251, Val Loss: 0.7788, Val Acc: 0.6667
Epoch 5/30, Train Loss: 0.0924, Val Loss: 0.8037, Val Acc: 0.6606
Epoch 6/30, Train Loss: 0.0669, Val Loss: 0.8529, Val Acc: 0.6727
Epoch 7/30, Train Loss: 0.0561, Val Loss: 0.8859, Val Acc: 0.6667
Epoch 8/30, Train Loss: 0.0438, Val Loss: 0.9074, Val Acc: 0.6788
Epoch 9/30, Train Loss: 0.0419, Val Loss: 0.9201, Val Acc: 0.6727
Epoch 10/30, Train Loss: 0.0357, Val Loss: 0.9205, Val Acc: 0.6667
Epoch 11/30, Train Loss: 0.0286, Val Loss: 0.8985, Val Acc: 0.6667
Epoch 12/30, Train Loss: 0.0312, Val Loss: 0.9275, Val Acc: 0.6667
Epoch 13/30, Train Loss: 0.0303, Val Loss: 0.9657, Val Acc: 0.6606
Epoch 14/30, Train Loss: 0.0278, Val Loss: 0.9444, Val Acc: 0.6788
Early stopp

In [ ]:
#finding tfidf params
def tune_tfidf_per_column(df, y_all, X_ord, X_cat):
    """
    Tune TF-IDF parameters separately for each text column
    """
    columns = ["best_tasks_free", "subopt_tasks_free", "verify_method_free"]
    
    # Start with your current config
    current_config = {
        "best_tasks_free": {"ngram_range": (1, 2), "max_features": 800, "min_df": 2, "max_df": 0.75},
        "subopt_tasks_free": {"ngram_range": (1, 2), "max_features": 500, "min_df": 2, "max_df": 0.75},
        "verify_method_free": {"ngram_range": (1, 1), "max_features": 300, "min_df": 2, "max_df": 0.75},
    }
    
    best_config = current_config.copy()
    
    # Parameter options to try
    param_options = {
        'ngram_range': [(1, 1), (1, 2), (1, 3)],
        'max_features': [300, 500, 800, 1000, 1200],
        'min_df': [1, 2, 3],
        'max_df': [0.7, 0.75, 0.8, 0.9]
    }
    
    # Tune each column one at a time
    for col in columns:
        print(f"\n{'='*60}")
        print(f"TUNING TF-IDF FOR COLUMN: {col}")
        print(f"{'='*60}")
        print(f"Current config: {current_config[col]}")
        
        best_acc_for_col = 0
        best_params_for_col = current_config[col]
        
        results = []
        
        # Try all combinations for this column
        for ngram in param_options['ngram_range']:
            for max_feat in param_options['max_features']:
                for min_df in param_options['min_df']:
                    for max_df in param_options['max_df']:
                        
                        test_config = {
                            'ngram_range': ngram,
                            'max_features': max_feat,
                            'min_df': min_df,
                            'max_df': max_df
                        }
                        
                        # Use best_config for other columns, test_config for current column
                        TEXT_TFIDF_CONFIG_TEST = best_config.copy()
                        TEXT_TFIDF_CONFIG_TEST[col] = test_config
                        
                        # Recompute features
                        X_text, _ = fit_tfidf_separate_columns(df, TEXT_TFIDF_CONFIG_TEST)
                        X_all_test = np.hstack([X_text, X_ord, X_cat])
                        
                        # Split
                        X_train_test, X_val_test, y_train_test, y_val_test = train_val_split(
                            X_all_test, y_all, test_size=0.2, random_state=42, stratify=True
                        )
                        
                        # Quick evaluation
                        np.random.seed(42)
                        model = NeuralNetwork(
                            input_size=X_train_test.shape[1],
                            hidden_units=384,  # Use best NN params
                            num_classes=num_classes,
                            dropout_rate=0.5,
                            lr=0.0003
                        )
                        
                        history = train_model(
                            model, X_train_test, y_train_test, 
                            X_val_test, y_val_test,
                            epochs=20, batch_size=32, patience=5
                        )
                        
                        val_loss, val_acc = evaluate_model(model, X_val_test, y_val_test, 32)
                        
                        results.append({
                            'config': test_config,
                            'val_acc': val_acc,
                            'feature_size': X_all_test.shape[1]
                        })
                        
                        if val_acc > best_acc_for_col:
                            best_acc_for_col = val_acc
                            best_params_for_col = test_config
        
        # Update best_config with the best parameters for this column
        best_config[col] = best_params_for_col
        
        # Sort and show top results for this column
        results.sort(key=lambda x: x['val_acc'], reverse=True)
        
        print(f"\nTop 3 configs for {col}:")
        for i, result in enumerate(results[:3]):
            print(f"{i+1}. {result['config']}")
            print(f"   Acc: {result['val_acc']:.4f}, Features: {result['feature_size']}")
        
        print(f"\nBest config selected: {best_params_for_col}")
        print(f"Best accuracy: {best_acc_for_col:.4f}")
    
    return best_config

# Run per-column tuning
print("Starting per-column TF-IDF tuning...")
print("This will take a while (testing 3×4×3×4 = 144 combinations per column)")

best_tfidf_per_column = tune_tfidf_per_column(df, y_all, X_ord, X_cat)

print(f"\n{'='*60}")
print("FINAL OPTIMIZED TF-IDF CONFIGURATION:")
print(f"{'='*60}")
for col, config in best_tfidf_per_column.items():
    print(f"{col}: {config}")

# Now use these optimized configs
X_text_final, _ = fit_tfidf_separate_columns(df, best_tfidf_per_column)
X_all_final = np.hstack([X_text_final, X_ord, X_cat])

X_train_final, X_val_final, y_train_final, y_val_final = train_val_split(
    X_all_final, y_all, test_size=0.2, random_state=42, stratify=True
)

print(f"\nFinal feature matrix shape: {X_all_final.shape}")

Starting per-column TF-IDF tuning...
This will take a while (testing 3×4×3×4 = 144 combinations per column)

TUNING TF-IDF FOR COLUMN: best_tasks_free
Current config: {'ngram_range': (1, 2), 'max_features': 800, 'min_df': 2, 'max_df': 0.75}
[best_tasks_free] TF-IDF shape: (825, 300), vocab size: 300
[subopt_tasks_free] TF-IDF shape: (825, 500), vocab size: 500
[verify_method_free] TF-IDF shape: (825, 300), vocab size: 300
Combined text TF-IDF shape: (825, 1100)
Epoch 1/20, Train Loss: 1.4132, Val Loss: 1.0483, Val Acc: 0.5212
Epoch 2/20, Train Loss: 0.9208, Val Loss: 0.9528, Val Acc: 0.6000
Epoch 3/20, Train Loss: 0.7650, Val Loss: 0.9134, Val Acc: 0.6303
Epoch 4/20, Train Loss: 0.6318, Val Loss: 0.9237, Val Acc: 0.6242
Epoch 5/20, Train Loss: 0.5442, Val Loss: 0.8911, Val Acc: 0.6424
Epoch 6/20, Train Loss: 0.4575, Val Loss: 0.8737, Val Acc: 0.6485
Epoch 7/20, Train Loss: 0.4465, Val Loss: 0.8831, Val Acc: 0.6424
Epoch 8/20, Train Loss: 0.4054, Val Loss: 0.8654, Val Acc: 0.6424
Epoch 

In [ ]:
# getting stats over multiple trials
def run_multiple_trials_with_best_params(n_trials=10):
    """Run multiple trials with best hyperparameters to assess stability"""
    results = []
    
    best_params = {
        'hidden_units': 512,
        'dropout_rate': 0.5,
        'lr': 0.0001,
        'batch_size': 64
    }
    
    print(f"Running {n_trials} trials with best hyperparameters...")
    print(f"Params: {best_params}\n")
    
    for trial in range(n_trials):
        np.random.seed(42 + trial)
        
        model = NeuralNetwork(
            input_size=X_train.shape[1],
            hidden_units=best_params['hidden_units'],
            num_classes=num_classes,
            dropout_rate=best_params['dropout_rate'],
            lr=best_params['lr']
        )
        
        history = train_model(
            model=model,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            epochs=30,
            batch_size=best_params['batch_size'],
            patience=10
        )
        
        val_loss, val_acc = evaluate_model(model, X_val, y_val, batch_size=64)
        
        # Per-class accuracy
        val_logits = model.forward(X_val, training=False)
        val_predictions = np.argmax(val_logits, axis=1)
        
        per_class_acc = {}
        for i, class_name in enumerate(unique_classes):
            mask = y_val == i
            if np.sum(mask) > 0:
                class_acc = np.sum(val_predictions[mask] == y_val[mask]) / np.sum(mask)
                per_class_acc[class_name] = class_acc
        
        results.append({
            'trial': trial + 1,
            'val_acc': val_acc,
            'val_loss': val_loss,
            'per_class': per_class_acc
        })
        
        print(f"Trial {trial + 1}: Val Acc = {val_acc:.4f}, Loss = {val_loss:.4f}")
    
    # Calculate statistics
    accuracies = [r['val_acc'] for r in results]
    losses = [r['val_loss'] for r in results]
    
    print(f"\n{'='*60}")
    print("SUMMARY STATISTICS")
    print(f"{'='*60}")
    print(f"Overall Accuracy:")
    print(f"  Mean: {np.mean(accuracies):.4f}")
    print(f"  Std:  {np.std(accuracies):.4f}")
    print(f"  Min:  {np.min(accuracies):.4f}")
    print(f"  Max:  {np.max(accuracies):.4f}")
    print(f"  Median: {np.median(accuracies):.4f}")
    
    print(f"\nValidation Loss:")
    print(f"  Mean: {np.mean(losses):.4f}")
    print(f"  Std:  {np.std(losses):.4f}")
    
    # Per-class statistics
    print(f"\n{'='*60}")
    print("PER-CLASS ACCURACY (averaged across trials)")
    print(f"{'='*60}")
    for class_name in unique_classes:
        class_accs = [r['per_class'][class_name] for r in results]
        print(f"{class_name}:")
        print(f"  Mean: {np.mean(class_accs):.4f} ± {np.std(class_accs):.4f}")
        print(f"  Range: [{np.min(class_accs):.4f}, {np.max(class_accs):.4f}]")
    
    # Find best trial
    best_trial = max(results, key=lambda x: x['val_acc'])
    print(f"\n{'='*60}")
    print(f"BEST TRIAL: #{best_trial['trial']}")
    print(f"{'='*60}")
    print(f"Accuracy: {best_trial['val_acc']:.4f}")
    print(f"Loss: {best_trial['val_loss']:.4f}")
    print("Per-class accuracy:")
    for class_name, acc in best_trial['per_class'].items():
        print(f"  {class_name}: {acc:.4f}")
    
    return results

# Run the trials
results = run_multiple_trials_with_best_params(n_trials=10)

Running 10 trials with best hyperparameters...
Params: {'hidden_units': 512, 'dropout_rate': 0.5, 'lr': 0.0001, 'batch_size': 64}

Epoch 1/30, Train Loss: 1.5082, Val Loss: 1.0952, Val Acc: 0.4000
Epoch 2/30, Train Loss: 1.2360, Val Loss: 1.0641, Val Acc: 0.4970
Epoch 3/30, Train Loss: 0.9778, Val Loss: 1.0164, Val Acc: 0.5455
Epoch 4/30, Train Loss: 0.8610, Val Loss: 0.9677, Val Acc: 0.6000
Epoch 5/30, Train Loss: 0.7607, Val Loss: 0.9329, Val Acc: 0.6182
Epoch 6/30, Train Loss: 0.7197, Val Loss: 0.9192, Val Acc: 0.6121
Epoch 7/30, Train Loss: 0.6233, Val Loss: 0.9025, Val Acc: 0.6061
Epoch 8/30, Train Loss: 0.5242, Val Loss: 0.8919, Val Acc: 0.6364
Epoch 9/30, Train Loss: 0.4864, Val Loss: 0.8829, Val Acc: 0.6303
Epoch 10/30, Train Loss: 0.5390, Val Loss: 0.8697, Val Acc: 0.6424
Epoch 11/30, Train Loss: 0.4583, Val Loss: 0.8565, Val Acc: 0.6364
Epoch 12/30, Train Loss: 0.4177, Val Loss: 0.8479, Val Acc: 0.6545
Epoch 13/30, Train Loss: 0.4251, Val Loss: 0.8547, Val Acc: 0.6545
Epoch 1

In [277]:
X_train, X_val, y_train, y_val = train_val_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=True
)

# X_all: from your tfidf + ordinal + categorical pipeline
# y: label-encoded targets (0,1,2)
# Suppose you've already done something like:
# X_train, X_val, y_train, y_val = train_test_split(X_all, y, test_size=0.2, random_state=42)

input_dim = X_train.shape[1]

model = ManualNNAdam(
    input_dim=input_dim,
    hidden_dim=256,
    output_dim=3,
    lr=1e-4,
    dropout_rate=0.4,
    reg_lambda=1e-4,
    batch_size=64,
    epochs=20,
    seed=42,
)

model.fit(X_train, y_train, X_val=X_val, y_val=y_val, verbose=True)

print("Final train acc:", model.accuracy(X_train, y_train))
print("Final val acc:  ", model.accuracy(X_val, y_val))



Epoch 1/20 - Train Loss: 1.1552 | Val Loss: 1.1307 | Val Acc: 0.3455
Epoch 2/20 - Train Loss: 1.1129 | Val Loss: 1.1014 | Val Acc: 0.4848
Epoch 3/20 - Train Loss: 1.0773 | Val Loss: 1.0796 | Val Acc: 0.5091
Epoch 4/20 - Train Loss: 1.0375 | Val Loss: 1.0589 | Val Acc: 0.5152
Epoch 5/20 - Train Loss: 1.0063 | Val Loss: 1.0398 | Val Acc: 0.5455
Epoch 6/20 - Train Loss: 0.9638 | Val Loss: 1.0209 | Val Acc: 0.5697
Epoch 7/20 - Train Loss: 0.9338 | Val Loss: 1.0037 | Val Acc: 0.5758
Epoch 8/20 - Train Loss: 0.9082 | Val Loss: 0.9876 | Val Acc: 0.5879
Epoch 9/20 - Train Loss: 0.8768 | Val Loss: 0.9723 | Val Acc: 0.5758
Epoch 10/20 - Train Loss: 0.8504 | Val Loss: 0.9575 | Val Acc: 0.5818
Epoch 11/20 - Train Loss: 0.8215 | Val Loss: 0.9431 | Val Acc: 0.5879
Epoch 12/20 - Train Loss: 0.7935 | Val Loss: 0.9285 | Val Acc: 0.5939
Epoch 13/20 - Train Loss: 0.7663 | Val Loss: 0.9150 | Val Acc: 0.6000
Epoch 14/20 - Train Loss: 0.7320 | Val Loss: 0.9022 | Val Acc: 0.6242
Epoch 15/20 - Train Loss: 0.7

In [ ]:
# bagging models
def train_ensemble(n_models=10):
    """Train ensemble and return predictions"""
    models = []
    best_params = {'hidden_units': 256, 'dropout_rate': 0.1, 'lr': 0.0001, 'batch_size': 64}
    #best_params = {'hidden_units': 512, 'dropout_rate': 0.5, 'lr': 0.0001, 'batch_size': 64}

    print(f"Training ensemble of {n_models} models...")
    for i in range(n_models):
        np.random.seed(42 + i)
        
        model = NeuralNetwork(
            input_size=X_train.shape[1],
            hidden_units=best_params['hidden_units'],
            num_classes=num_classes,
            dropout_rate=best_params['dropout_rate'],
            lr=best_params['lr']
        )
        
        print(f"\nModel {i+1}/{n_models}:")
        train_model(model, X_train, y_train, X_val, y_val, 
                   epochs=30, batch_size=best_params['batch_size'], patience=10)
        models.append(model)
    
    return models

# Train ensemble
ensemble_models = train_ensemble(n_models=20)

# Ensemble predictions (soft voting - average probabilities)
all_logits = [model.forward(X_val, training=False) for model in ensemble_models]
avg_logits = np.mean(all_logits, axis=0)
ensemble_preds = np.argmax(avg_logits, axis=1)
ensemble_acc = np.mean(ensemble_preds == y_val)

print(f"\n{'='*60}")
print("ENSEMBLE RESULTS")
print(f"{'='*60}")
print(f"Ensemble Accuracy: {ensemble_acc:.4f}")

# Per-class accuracy
for i, class_name in enumerate(unique_classes):
    mask = y_val == i
    if np.sum(mask) > 0:
        class_acc = np.sum(ensemble_preds[mask] == y_val[mask]) / np.sum(mask)
        print(f"{class_name}: {class_acc:.4f}")

Training ensemble of 20 models...

Model 1/20:
Epoch 1/30, Train Loss: 1.3754, Val Loss: 1.0902, Val Acc: 0.4848
Epoch 2/30, Train Loss: 1.0999, Val Loss: 1.0709, Val Acc: 0.5212
Epoch 3/30, Train Loss: 0.8671, Val Loss: 1.0363, Val Acc: 0.5455
Epoch 4/30, Train Loss: 0.7569, Val Loss: 1.0054, Val Acc: 0.5879
Epoch 5/30, Train Loss: 0.6443, Val Loss: 0.9681, Val Acc: 0.5939
Epoch 6/30, Train Loss: 0.5891, Val Loss: 0.9343, Val Acc: 0.5939
Epoch 7/30, Train Loss: 0.4991, Val Loss: 0.8996, Val Acc: 0.6061
Epoch 8/30, Train Loss: 0.4567, Val Loss: 0.8756, Val Acc: 0.6242
Epoch 9/30, Train Loss: 0.4189, Val Loss: 0.8503, Val Acc: 0.6545
Epoch 10/30, Train Loss: 0.3737, Val Loss: 0.8396, Val Acc: 0.6364
Epoch 11/30, Train Loss: 0.3546, Val Loss: 0.8338, Val Acc: 0.6485
Epoch 12/30, Train Loss: 0.3013, Val Loss: 0.8224, Val Acc: 0.6485
Epoch 13/30, Train Loss: 0.2976, Val Loss: 0.8195, Val Acc: 0.6545
Epoch 14/30, Train Loss: 0.2731, Val Loss: 0.8086, Val Acc: 0.6667
Epoch 15/30, Train Loss:

In [135]:
# Recommended: 3-5 trials per combination for reliable estimates
param_grid = {
    'hidden_units': [256, 384, 512, 768],
    'dropout_rate': [0.3, 0.4, 0.5, 0.6],
    'lr': [0.00005, 0.0001, 0.0003, 0.0005],
    'batch_size': [32, 64]
}

# This will run 4×4×4×2 = 128 combinations × 5 trials = 640 training runs
# Takes a while but gives you the TRUE best hyperparameters
results = grid_search(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    input_size=X_train.shape[1],
    num_classes=num_classes,
    param_grid=param_grid,
    epochs=30,
    patience=10,
    n_trials=5  # Run 5 trials per combination and average
)

# Get the best parameters (based on average performance)
best_params = results[0]['params']
print(f"\nBest hyperparameters (averaged over 5 trials): {best_params}")
print(f"Expected accuracy: {results[0]['avg_val_acc']:.4f} ± {results[0]['std_val_acc']:.4f}")

Testing 128 hyperparameter combinations with 5 trial(s) each...
Total training runs: 640

[1/128] Testing: {'hidden_units': 256, 'dropout_rate': 0.3, 'lr': 5e-05, 'batch_size': 32}
  Trial 1/5... Epoch 1/30, Train Loss: 1.2177, Val Loss: 1.0115, Val Acc: 0.4970
Epoch 2/30, Train Loss: 1.1013, Val Loss: 0.9329, Val Acc: 0.5152
Epoch 3/30, Train Loss: 1.0148, Val Loss: 0.9087, Val Acc: 0.5455
Epoch 4/30, Train Loss: 0.9681, Val Loss: 0.8877, Val Acc: 0.5515
Epoch 5/30, Train Loss: 0.9014, Val Loss: 0.8631, Val Acc: 0.5879
Epoch 6/30, Train Loss: 0.8875, Val Loss: 0.8552, Val Acc: 0.5818
Epoch 7/30, Train Loss: 0.7991, Val Loss: 0.8426, Val Acc: 0.6061
Epoch 8/30, Train Loss: 0.7488, Val Loss: 0.8252, Val Acc: 0.6364
Epoch 9/30, Train Loss: 0.7202, Val Loss: 0.8280, Val Acc: 0.6182
Epoch 10/30, Train Loss: 0.6947, Val Loss: 0.8134, Val Acc: 0.6364
Epoch 11/30, Train Loss: 0.6938, Val Loss: 0.8051, Val Acc: 0.6545
Epoch 12/30, Train Loss: 0.6626, Val Loss: 0.8001, Val Acc: 0.6606
Epoch 13/

In [ ]:
# TF=IDF DEVELOPMENT IN ISOLATION
"""CORPUS:
├── Document 1 (Row 1): col1 + col2 + col3 → Label: ChatGPT
├── Document 2 (Row 2): col1 + col2 + col3 → Label: Claude 

Term frequency (TF): the raw count of a term in a document
Inverse document frequency (IDF): how important a word is, idf(t,D)=log(N/df(t))
"""
# GridSearchCV for hyperparam tuning
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: TAKE OUT WHEN YOU MOVE TO PIPELINE 
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df[text_cols] = imputer.fit_transform(df[text_cols])

# make corpus, create new column
df['corpus'] = df[text_cols].agg(' '.join, axis=1)
corpus = df['corpus']
labels = df['target']

# Create pipeline
pipeline = Pipeline([
    # TODO: also try removing stop words and adding max_df
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__max_features': [500, 1000, 2000, 3000, None], # max number of features 
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)], # length of context range 
    'tfidf__min_df': [1, 2, 3, 4, 5] # minimum number times word has to appear to be included
}

# Search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(corpus, labels)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy: {grid_search.best_score_:.3f}")

In [ ]:
# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1) 
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# embed_best_tasks_free = embed_texts(df['best_tasks_free'])
# print(embed_best_tasks_free)
# # embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])

def embed_texts(text_cols, df, batch_size=32):
    col_embedding = []
    for col in text_cols:
        embed_texts(df[col], batch_size=batch_size)
        col_embedding.append(col)
    return np.hstack(col_embedding)
     
print(embed_texts(text_cols, df))


c:\Users\carol\Desktop\Characterize_GenAIModel\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:

def split_data(df):
    students = df['id'].unique()
    train_df, test_df = train_test_split(students, test_size=0.3, random_state=42)
